# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arjelmilan/flyrank-ai/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

The goal is to assign each content page a priority score so that content/SEO specialists can review the most important pages first. Ranking is more appropriate than simple classification because the final decision is not just whether a page needs attention; it is which pages should be reviewed first when review capacity is limited.

In [2]:
# Check the target-related fields available in the starter data

import pandas as pd

url = "https://raw.githubusercontent.com/arjelmilan/flyrank-ai/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Shape:", df.shape)
print("\nTrend distribution:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nTrend proportions:")
print(df["trend_direction"].value_counts(normalize=True, dropna=False))

Shape: (30000, 44)

Trend distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend proportions:
trend_direction
down      0.542067
stable    0.198733
up        0.146267
new       0.074533
flat      0.038400
Name: proportion, dtype: float64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*
For the starter experiment, I will use trend_direction == "down" as the target/proxy. This label comes from the dataset's existing rule rather than from a future observed outcome.

Therefore, the model can measure how well the available signals distinguish pages currently labeled as declining, but this should not be described as predicting future decline. In later work, I will investigate whether a future-window outcome can be constructed to make the target more directly aligned with the refresh decision.

In [3]:
# Define the starter proxy target

df["target_down"] = (df["trend_direction"] == "down").astype(int)

print("Target distribution:")
print(df["target_down"].value_counts())

print("\nPositive rate:",
      round(df["target_down"].mean() * 100, 2), "%")

Target distribution:
target_down
1    16262
0    13738
Name: count, dtype: int64

Positive rate: 54.21 %


## 3. Success metric

*One metric you can defend. What number means 'good'?*
I will use Precision@50 as the primary metric because the practical decision is to identify the highest-priority pages for human review. A good model should place a high proportion of relevant pages among the first 50 recommendations.

I will compare the model against a simple rule-based baseline. Higher Precision@50 means the review team can spend its limited attention on a more relevant set of pages.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
k = 50

positive_rate = df["target_down"].mean()

print(f"Baseline positive rate: {positive_rate:.3f}")
print(f"Expected positives in a random top-{k}: "
      f"{positive_rate * k:.1f}")

Baseline positive rate: 0.542
Expected positives in a random top-50: 27.1


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Each row represents one content page and contains its observed search, engagement, content-age, and trend-related signals over the available observation window. The model will assign a priority score to each page, which can then be used to create a ranked review queue.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Show the dataframe and verify one row corresponds to one page

print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

display(df.head())

# Check whether the page identifier is unique
print("\nUnique page IDs:")

if "content_id" in df.columns:
    print(df["content_id"].nunique(), "unique page IDs")
else:
    print("content_id column not found; inspect the columns above.")


Number of rows: 30000
Number of columns: 45


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,target_down
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1



Unique page IDs:
30000 unique page IDs


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Page priority depends on several signals at the same time: search visibility, sessions, CTR, ranking position, content age, engagement, and trend. A fixed rule would require manually choosing thresholds and deciding how these signals should be combined.

ML can learn combinations of these signals from historical examples and produce a continuous priority score. This is useful because two declining pages may have very different business importance: a page with high impressions and strong historical demand may deserve attention before a low-traffic page with the same decline label.

I will therefore compare ML against a transparent rule-based baseline rather than assuming that ML is automatically better.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.